In [17]:
pip install PyCO2SYS

Note: you may need to restart the kernel to use updated packages.


In [18]:
# SOCAT Carbonate System Calculations Notebook

import pandas as pd
import PyCO2SYS as pyco2

In [19]:
# 1. Load the SOCAT dataset with AT_estimate
file_path = "/Users/Mbongeleni/Library/CloudStorage/OneDrive-StellenboschUniversity/Documents/CSIR/SOCAT/SOCAT_combined.xlsx"
df = pd.read_excel(file_path)

In [20]:
# 2. Prepare input variables
# AT_estimate is in µmol/kg, SAL is practical salinity, TEMP is in °C,
# and pCO2 (µatm) is available in the dataset as 'FCO2_RECOMMENDED'.

sss = df["SAL"].values        # Practical salinity
sst = df["TEMP"].values       # Temperature in °C
alk = df["AT_estimate"].values      # Total Alkalinity (µmol/kg)
fco2 = df["FCO2_RECOMMENDED"].values # Fugacity of CO2 (µatm)

# Converting pressures from hPa to atm (1 atm = 1013.25 hPa)
press_atm = df["PRESSURE_ATM"].values / 1013.25
press_equi = df["PRESSURE_EQUI"].values / 1013.25

In [21]:
# 3. Run PyCO2SYS calculations
# Inputs: Alk and pCO2, with salinity and temperature
results = pyco2.sys(
    par1=alk,
    par2=fco2,
    par1_type=1,
    par2_type=5,
    salinity=sss,
    temperature=sst,
    temperature_out=sst,
    pressure=press_equi,       # equilibrator chamber pressure
    pressure_out=press_atm,    # sea-level atmospheric pressure
    opt_pH_scale=1,            # pH on total scale
    opt_k_carbonic=14,         # Southern Ocean recommended constants
    opt_k_bisulfate=2          # Southern Ocean recommended constants
)

/opt/anaconda3/lib/python3.13/site-packages/autograd/tracer.py:54: RuntimeWarning: invalid value encountered in sqrt
  return f_raw(*args, **kwargs)
/opt/anaconda3/lib/python3.13/site-packages/PyCO2SYS/equilibria/p1atm.py:84: RuntimeWarning: invalid value encountered in sqrt
  * Sal**0.5
/opt/anaconda3/lib/python3.13/site-packages/PyCO2SYS/equilibria/p1atm.py:86: RuntimeWarning: invalid value encountered in power
  + (-0.00291179 + 0.0000209968 * TempK) * Sal**1.5
/opt/anaconda3/lib/python3.13/site-packages/PyCO2SYS/equilibria/p1atm.py:99: RuntimeWarning: invalid value encountered in sqrt
  lnKF = 1590.2 / TempK - 12.641 + 1.525 * IonS**0.5
/opt/anaconda3/lib/python3.13/site-packages/PyCO2SYS/equilibria/p1atm.py:113: RuntimeWarning: invalid value encountered in sqrt
  lnKF = 874 / TempK - 9.68 + 0.111 * Sal**0.5
/opt/anaconda3/lib/python3.13/site-packages/PyCO2SYS/equilibria/p1atm.py:479: RuntimeWarning: invalid value encountered in sqrt
  - 0.071692 * F1 * Sal**0.5
/opt/anaconda3/lib/

In [22]:
# 4. Extract desired outputs
df["CT_calculated"] = results["dic"]          # Dissolved inorganic carbon (µmol/kg)
df["pH_total"] = results["pH_out"]            # pH on total scale
df["Omega_arag"] = results["saturation_aragonite"]    # Aragonite saturation state


In [23]:
# 5. Save updated dataset back to Excel
df.to_excel(file_path, index=False)

print("CT_calculated, pH_total, and Omega_arag successfully added to SOCAT_combined.xlsx")


CT_calculated, pH_total, and Omega_arag successfully added to SOCAT_combined.xlsx
